# 🚛 Concrete Mixer Truck Detection System
## Production-Ready YOLO26n-OBB - Local Execution

---

### 📋 Overview

**Complete Pipeline for Local Development:**
1. 📦 Environment Check
2. 📊 Data Preparation & Fusion
3. 🎓 Model Training with MLflow
4. ✅ Model Validation
5. 🎬 Video Processing
6. 📈 Results Visualization

**Features:**
- YOLO26n-OBB for oriented bounding boxes
- Optical Flow + SSIM rotation detection
- State machine (POURING/IN_TRANSIT/IDLE)
- MLflow experiment tracking

---

### ⚙️ Requirements
- Python 3.10+
- CUDA GPU (recommended)
- ~10GB disk space
- Dependencies installed (`pip install -r requirements.txt`)

## 📦 Phase 0: Environment Check

Verify system setup and dependencies

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

In [ ]:
# Check CUDA availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CUDA not available - training will be slower on CPU")

In [ ]:
# Check working directory
import os
from pathlib import Path

current_dir = Path.cwd()
print(f"Current directory: {current_dir}")
print(f"\nProject files:")
for item in ['src', 'data', 'config.yaml', 'main.py', 'requirements.txt']:
    exists = (current_dir / item).exists()
    status = "✅" if exists else "❌"
    print(f"  {status} {item}")

## 📊 Phase 1: Data Preparation

Load and merge datasets using production modules

In [ ]:
# Add src to path
import sys
from pathlib import Path

src_path = Path.cwd() / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"✅ Added to path: {src_path}")

In [ ]:
# Import production modules
from config import get_config
from data_loader import DataLoader

print("✅ Modules imported successfully!")

In [ ]:
# Load configuration
config = get_config(yaml_path='config.yaml')

print("="*70)
print("CONFIGURATION LOADED")
print("="*70)
print(f"📋 Project: {config['project'].PROJECT_NAME}")
print(f"🎯 Model: {config['model'].MODEL_NAME}")
print(f"📊 Classes: {config['dataset'].CLASSES}")
print(f"🔄 Batch Size: {config['model'].BATCH_SIZE}")
print(f"📈 Epochs: {config['model'].EPOCHS}")
print(f"🎨 Augmentation:")
print(f"   - Rotation: ±{config['model'].DEGREES}°")
print(f"   - Mosaic: {config['model'].MOSAIC}")
print(f"   - MixUp: {config['model'].MIXUP}")
print("="*70)

In [ ]:
# Initialize data loader
loader = DataLoader(config['dataset'])

# Load all datasets
dataset1, dataset2 = loader.load_all_datasets()

print(f"\n✅ Datasets loaded!")
print(f"   Dataset 1 (Mixer): {'✓' if dataset1 else '✗'}")
print(f"   Dataset 2 (Concrete Mixed Truck): {'✓' if dataset2 else '✗'}")

In [ ]:
# Merge datasets
merged_dir = loader.merge_datasets(
    [dataset1, dataset2],
    config['dataset'].MERGED_DIR
)

# Create data.yaml
data_yaml_path = loader.create_data_yaml(
    merged_dir,
    config['dataset'].CLASSES,
    config['dataset'].NUM_CLASSES
)

print(f"\n✅ Data preparation complete!")
print(f"   Merged dataset: {merged_dir}")
print(f"   Data YAML: {data_yaml_path}")

In [ ]:
# Visualize dataset statistics
import matplotlib.pyplot as plt
from pathlib import Path

train_images = list((merged_dir / 'train' / 'images').glob('*'))
train_labels = list((merged_dir / 'train' / 'labels').glob('*.txt'))
valid_images = list((merged_dir / 'valid' / 'images').glob('*'))
valid_labels = list((merged_dir / 'valid' / 'labels').glob('*.txt'))

print(f"\n📊 Dataset Statistics:")
print(f"   Training images: {len(train_images)}")
print(f"   Training labels: {len(train_labels)}")
print(f"   Validation images: {len(valid_images)}")
print(f"   Validation labels: {len(valid_labels)}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Train/Val split
splits = ['Train', 'Valid']
counts = [len(train_images), len(valid_images)]
colors = ['#3498db', '#2ecc71']
ax1.bar(splits, counts, color=colors)
ax1.set_ylabel('Image Count', fontsize=12)
ax1.set_title('Dataset Split', fontsize=14, fontweight='bold')
for i, v in enumerate(counts):
    ax1.text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=12)

# Images vs Labels
categories = ['Train\nImages', 'Train\nLabels', 'Valid\nImages', 'Valid\nLabels']
counts2 = [len(train_images), len(train_labels), len(valid_images), len(valid_labels)]
colors2 = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
ax2.bar(categories, counts2, color=colors2)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Images vs Labels', fontsize=14, fontweight='bold')
for i, v in enumerate(counts2):
    ax2.text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 🎓 Phase 2: Model Training

Train YOLO26n-OBB with MLflow tracking

In [ ]:
# Import trainer
from trainer import ModelTrainer

# Initialize trainer
trainer = ModelTrainer(
    config['model'],
    config['mlflow'],
    config['paths']
)

print("✅ Trainer initialized!")

In [ ]:
# Setup MLflow
trainer.setup_mlflow()

print("\n✅ MLflow tracking configured!")
print(f"   Tracking URI: {trainer.mlflow_config.TRACKING_DIR}")
print(f"   Experiment: {trainer.mlflow_config.EXPERIMENT_NAME}")

In [ ]:
# Load model
model = trainer.load_model()

if model:
    print("✅ Model loaded successfully!")
    print(f"   Model: {config['model'].MODEL_NAME}")
    print(f"   Type: {type(model).__name__}")
else:
    print("❌ Failed to load model")
    print("   Make sure yolo26n-obb.pt is available")

In [ ]:
# Train model
print("="*70)
print("🎓 STARTING TRAINING")
print("="*70)
print(f"   Dataset: {data_yaml_path}")
print(f"   Epochs: {config['model'].EPOCHS}")
print(f"   Batch size: {config['model'].BATCH_SIZE}")
print(f"   Image size: {config['model'].INPUT_SIZE}")
print("="*70)
print("\n⏳ This may take several hours...\n")

best_model_path, final_metrics = trainer.train(str(data_yaml_path))

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"   Best model: {best_model_path}")
print(f"   Final metrics:")
for metric, value in final_metrics.items():
    print(f"      {metric}: {value:.4f}")
print("="*70)

In [ ]:
# Visualize training results
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load results
results_path = Path(config['paths'].RUNS_DIR) / config['paths'].MODEL_NAME / 'results.csv'

if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    
    # Create 2x2 subplot
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Training Metrics', fontsize=16, fontweight='bold')
    
    # mAP50
    if 'metrics/mAP50(B)' in df.columns:
        axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], 'b-', linewidth=2.5)
        axes[0, 0].set_title('mAP@0.5', fontsize=13, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch', fontsize=11)
        axes[0, 0].set_ylabel('mAP50', fontsize=11)
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].set_ylim([0, 1])
    
    # mAP50-95
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], 'g-', linewidth=2.5)
        axes[0, 1].set_title('mAP@0.5:0.95', fontsize=13, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch', fontsize=11)
        axes[0, 1].set_ylabel('mAP50-95', fontsize=11)
        axes[0, 1].grid(True, alpha=0.3)
        axes[0, 1].set_ylim([0, 1])
    
    # Box Loss
    if 'train/box_loss' in df.columns:
        axes[1, 0].plot(df['epoch'], df['train/box_loss'], 'r-', linewidth=2.5, label='Train')
        if 'val/box_loss' in df.columns:
            axes[1, 0].plot(df['epoch'], df['val/box_loss'], 'orange', linewidth=2.5, label='Val', linestyle='--')
        axes[1, 0].set_title('Box Loss', fontsize=13, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch', fontsize=11)
        axes[1, 0].set_ylabel('Loss', fontsize=11)
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].legend()
    
    # Class Loss
    if 'train/cls_loss' in df.columns:
        axes[1, 1].plot(df['epoch'], df['train/cls_loss'], 'purple', linewidth=2.5, label='Train')
        if 'val/cls_loss' in df.columns:
            axes[1, 1].plot(df['epoch'], df['val/cls_loss'], 'pink', linewidth=2.5, label='Val', linestyle='--')
        axes[1, 1].set_title('Classification Loss', fontsize=13, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch', fontsize=11)
        axes[1, 1].set_ylabel('Loss', fontsize=11)
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Training metrics visualized!")
    print(f"   Total epochs: {len(df)}")
    print(f"   Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
    print(f"   Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
else:
    print("⚠️ Results file not found")
    print(f"   Expected: {results_path}")

## ✅ Phase 3: Model Validation

Validate trained model on test set

In [ ]:
# Validate model
print("="*70)
print("✅ RUNNING VALIDATION")
print("="*70)

val_results = trainer.validate(best_model_path, str(data_yaml_path))

print("\n✅ Validation complete!")

## 🎬 Phase 4: Video Processing

Process video with rotation detection and state classification

In [ ]:
# Import video processor
from video_processor import VideoProcessor

# Initialize processor
processor = VideoProcessor(best_model_path, config['rotation'])

print("✅ Video processor initialized!")
print(f"   Model: {best_model_path}")
print(f"   Optical Flow threshold: {config['rotation'].FLOW_MAGNITUDE_THRESHOLD}")
print(f"   SSIM threshold: {config['rotation'].SSIM_THRESHOLD}")

In [ ]:
# Specify input video path
# Change this to your video file path
input_video = 'path/to/your/video.mp4'  # ← แก้ path ตรงนี้
output_video = 'output_analysis.mp4'

print(f"📹 Input video: {input_video}")
print(f"📹 Output video: {output_video}")

# Check if input exists
if not Path(input_video).exists():
    print(f"\n⚠️ Input video not found: {input_video}")
    print("   Please update the 'input_video' variable with correct path")
else:
    print(f"\n✅ Input video found!")

In [ ]:
# Process video (only if input exists)
if Path(input_video).exists():
    print("="*70)
    print("🎬 PROCESSING VIDEO")
    print("="*70)
    print("   This may take a while depending on video length...\n")
    
    stats = processor.process_video(input_video, output_video)
    
    print("\n" + "="*70)
    print("✅ VIDEO PROCESSING COMPLETE!")
    print("="*70)
    print(f"   Output saved: {output_video}")
else:
    print("⚠️ Skipping video processing - input not found")
    stats = None

In [ ]:
# Visualize statistics
if stats:
    import matplotlib.pyplot as plt
    
    # Plot state distribution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    states = list(stats.keys())
    counts = list(stats.values())
    colors = ['#e74c3c', '#f39c12', '#2ecc71', '#95a5a6', '#ecf0f1']
    
    bars = ax1.bar(states, counts, color=colors[:len(states)])
    ax1.set_ylabel('Frame Count', fontsize=13)
    ax1.set_title('Truck State Distribution (Bar Chart)', fontsize=14, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # Pie chart
    total_frames = sum(stats.values())
    percentages = [(count / total_frames) * 100 for count in counts]
    
    wedges, texts, autotexts = ax2.pie(counts, labels=states, autopct='%1.1f%%',
                                        colors=colors[:len(states)],
                                        startangle=90,
                                        textprops={'fontsize': 11, 'fontweight': 'bold'})
    ax2.set_title('Truck State Distribution (Pie Chart)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*70)
    print("📊 STATE SUMMARY")
    print("="*70)
    for state, count in stats.items():
        percentage = (count / total_frames) * 100 if total_frames > 0 else 0
        print(f"   {state:20s}: {count:6d} frames ({percentage:5.1f}%)")
    print("="*70)
    print(f"   Total frames: {total_frames}")
    print("="*70)
else:
    print("⚠️ No statistics to visualize")

## 📈 Phase 5: MLflow Results

View experiment tracking results

In [ ]:
# List MLflow experiments
import mlflow

mlflow.set_tracking_uri(f'file://{Path(config["mlflow"].TRACKING_DIR).absolute()}')

experiments = mlflow.search_experiments()

print("="*70)
print("📊 MLFLOW EXPERIMENTS")
print("="*70)
for exp in experiments:
    print(f"\n   Name: {exp.name}")
    print(f"   ID: {exp.experiment_id}")
    print(f"   Artifact Location: {exp.artifact_location}")
    print(f"   Lifecycle Stage: {exp.lifecycle_stage}")
print("="*70)

In [ ]:
# Start MLflow UI (in separate terminal)
print("="*70)
print("📈 MLFLOW UI")
print("="*70)
print("\nTo view MLflow UI, run this command in a separate terminal:")
print("\n   mlflow ui --backend-store-uri ./mlruns --port 5000")
print("\nThen open your browser at:")
print("\n   http://localhost:5000")
print("\n" + "="*70)

## 📦 Phase 6: Model Export

Export model to different formats

In [ ]:
# Export to ONNX
from ultralytics import YOLO

print("="*70)
print("📦 EXPORTING MODEL TO ONNX")
print("="*70)

model = YOLO(best_model_path)
onnx_path = model.export(format='onnx', dynamic=True, simplify=True)

print(f"\n✅ ONNX export complete!")
print(f"   Path: {onnx_path}")
print("="*70)

In [ ]:
# Summary of all outputs
print("="*70)
print("🎉 PIPELINE COMPLETE - SUMMARY")
print("="*70)
print("\n📁 Generated Files:")
print(f"\n   1. Merged Dataset:")
print(f"      {merged_dir}")
print(f"\n   2. Trained Model (PyTorch):")
print(f"      {best_model_path}")
print(f"\n   3. ONNX Model:")
print(f"      {onnx_path}")
if stats:
    print(f"\n   4. Processed Video:")
    print(f"      {output_video}")
print(f"\n   5. MLflow Tracking:")
print(f"      {config['mlflow'].TRACKING_DIR}")
print(f"\n   6. Training Results:")
print(f"      {config['paths'].RUNS_DIR}/{config['paths'].MODEL_NAME}/")
print("\n" + "="*70)
print("\n📊 Performance Metrics:")
for metric, value in final_metrics.items():
    print(f"   {metric}: {value:.4f}")
print("\n" + "="*70)
print("\n🚀 Next Steps:")
print("   1. Review MLflow UI: mlflow ui --backend-store-uri ./mlruns")
print("   2. Test ONNX model on edge devices")
print("   3. Deploy to production environment")
print("   4. Monitor real-time performance")
print("\n" + "="*70)